# Demo 1: Simple Write/Read with Actor ContextThis notebook teaches the foundational SDK pattern with actor identity: authenticate, create a memory, and verify it round-trips correctly.## Learning Goals1. Initialize `NinaiClient` and authenticate with credentials.2. Create a memory with actor context (who is writing, what role, what responsibility).3. Read it back by ID using `get()` with reader context.4. Verify actor metadata is preserved in the response.

## Step 1: Setup and Login

Run the next cell to initialize the SDK and authenticate.

Expected outcome:
- No errors.
- A `client` object is ready.
- You are authenticated with the demo account.

In [1]:
from ninai import NinaiClient
import uuid

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)

User(id='550e8400-e29b-41d4-a716-446655440100', email='demo@ninai.dev', full_name='Demo User', avatar_url=None, is_active=True, clearance_level=5, organization_id='550e8400-e29b-41d4-a716-446655440000', organization_name=None, roles=['org_admin'], created_at=datetime.datetime(2026, 4, 11, 3, 15, 4, 364009, tzinfo=TzInfo(0)), last_login_at=datetime.datetime(2026, 4, 14, 15, 6, 10, 274355), preferences={})

## Step 2: Create a Memory, Retrieve, and Verify

Run the next cell to:
1. Create a memory (actor context is automatically derived from your login session).
2. Read it back by ID using `get()`.
3. Verify the actor context is preserved in the response.

Expected outcome:
- `memory_id` is a unique UUID.
- `write_actor_id` is present in the response (derived from your authenticated user).
- `read_ok` is True (confirming the roundtrip worked).

In [2]:
from datetime import datetime, timezone

marker = f'demo1-{uuid.uuid4()}'
write_time_utc = datetime.now(timezone.utc)

# Create memory -- actor identity is resolved automatically from the authenticated session.
# Pass anonymous=True only if you want to suppress attribution for this specific write.
created = client.memories.create(
    content=marker,
    source_type='manual',
    occurred_at=write_time_utc,
    # anonymous=True  # uncomment to suppress actor attribution
)

# Read back
read_back = client.memories.get(created.id)

listed = client.memories.list(page_size=20)
in_list = any(m.id == created.id for m in listed.items)

print('memory_id:', created.id)
print('write_actor_id:', created.write_actor_id)  # set server-side from JWT
print('write_role:', created.write_role)            # from AD/SCIM if available
print('read_ok:', read_back.id == created.id)
print('listed_ok:', in_list)

memory_id: bb4e4426-b11a-4ac4-93a1-90ab6dde177d
read_ok: True
listed_ok: True


## Step 3: Understanding Actor Context

### What is Actor Context?

Actor context tracks **who** wrote a memory. As of the current API, actor identity is
**never caller-supplied** -- it is always resolved server-side from your authenticated session:

| Response field | Source |
|---|---|
| `write_actor_id` | `user_id` from the JWT (`sub` claim) |
| `write_actor_type` | `"employee"` for human users; `"bot"` for API-key sessions |
| `write_role` | Job title from AD/SCIM; falls back to `"employee"` if unavailable |
| `write_responsibility` | Department description from AD/SCIM (optional) |
| `write_identity_mode` | `"full"`, `"role_only"`, or `"anonymous"` per org policy |

### Why This Matters
1. **Tamper-proof** -- callers cannot spoof another employee's identity.
2. **Audit Trail** -- every memory is automatically attributed without extra code.
3. **Opt-out** -- pass `anonymous=True` on a single write to suppress attribution (blocked if the org mandates full identity).
4. **Downstream agents** -- `OrgAttentionAgent` and `SocialMemoryAgent` use these fields for credibility scoring and attention routing.

### JWT Claims in Play
The access token carries: `sub` (user_id), `org_id`, `roles`, and optionally `group_id` / `team_id`.
These flow through `TenantContext` middleware and `IdentityResolverService` before reaching the memory layer.

### Verification Checklist
- `memory_id` is a string (UUID format).
- `write_actor_id` matches your authenticated user, not a hardcoded string.
- `read_ok` is True.
- All three passed -- automatic actor-context attribution works.

### Next Steps
- Move to **Demo 2** to search with reader context.
- Move to **Demo 4** to see role-based briefing (leadership vs. engineering views).